# 2. Data Cleaning (V2 - updated by Nikhil)

**Input:** `data/eda_start/reports_level.parquet` (notebook 1, one row per FAERS case).

**Output** in `data/clean/`:

| File | One row per | Contents |
|---|---|---|
| `reports_clean.parquet` | case | decoded, cleaned case fields and the target `y_hosp_or_death` |
| `case_ingredients/` | case × suspect ingredient | standardized ingredient names |
| `case_reactions/` | case × reaction term | MedDRA preferred terms |
| `cleaning_log.csv` | cleaning step | rows before / after each step |

**Cleaning decisions**

1. **Target.** Rebuild death / hospitalization from the raw codes (notebook 1 turned "missing" into `False`).
2. **Quarter** names: `faers_xml_2022q1` → `2022Q1`.
3. **Sex** code `0` (unknown) → missing.
4. **Age group** from age in years, else from the reporter's age-group code.
5. **Weight** outside 0.3–350 kg (or > 40 kg under age 2) → missing.
6. **Drop** cases with no suspect drug.
7. **Duplicates:** drop small groups of identical cases; flag large groups as "mass reports".
8. **Ingredients:** split combination products, strip salt names (`RANITIDINE HYDROCHLORIDE` → `RANITIDINE`).

## 1. Setup

In [1]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

INPUT = Path("data/eda_start/reports_level.parquet")
OUT = Path("data/clean")
OUT.mkdir(parents=True, exist_ok=True)

WEIGHT_MIN, WEIGHT_MAX = 0.3, 350  # kg
DUP_MAX_GROUP = 5                  # identical groups up to this size = duplicates; larger = mass reports
BATCH_SIZE = 500_000               # rows per batch when reading the long text columns

log = []


def log_step(step, rows_before, rows_after, note=""):
    """Record how many rows a step removed (saved as cleaning_log.csv at the end)."""
    log.append({"step": step, "rows_before": rows_before, "rows_after": rows_after, "note": note})
    print(f"{step}: {rows_before:,} -> {rows_after:,}  {note}")

## 2. Load

We load every column **except** the two long text lists (`suspect_ingredients`, `reaction_pts`), which would need
~10 GB of memory. Those are read in batches in section 8.

`read_dictionary` loads the short code columns (`"1"`, `"2"`, …) as categoricals.

In [2]:
CODE_COLS = [
    "source_quarter", "reporttype", "serious", "seriousnessdeath", "seriousnesshospitalization",
    "fulfillexpeditecriteria", "qualification", "patientagegroup", "patientsex",
    "primarysourcecountry", "occurcountry", "receiptdate",
]
OTHER_COLS = [
    "safetyreportid", "safetyreportversion", "receivedate_parsed", "age_years", "patientweight_kg",
    "n_drugs", "n_suspect_drugs", "n_reactions",
]

df = pd.read_parquet(INPUT, columns=CODE_COLS + OTHER_COLS, read_dictionary=CODE_COLS)
df["safetyreportid"] = df["safetyreportid"].astype("int64")  # ids are 8-digit numbers

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns, {df.memory_usage(deep=True).sum() / 1e9:.1f} GB")
log_step("load", len(df), len(df))

8,555,729 rows x 20 columns, 0.7 GB
load: 8,555,729 -> 8,555,729  


## 3. Checks and quarter names

Each case id must appear once. 

In [ ]:
assert df["safetyreportid"].is_unique, "duplicate case ids: check notebook 1"

quarter = df["source_quarter"].astype(str).str.upper().str[-6:]  # 'faers_xml_2022q1' -> '2022Q1'
df["quarter"] = pd.Categorical(quarter, categories=sorted(quarter.unique()), ordered=True)

all_quarters = pd.period_range(quarter.min(), quarter.max(), freq="Q").astype(str)
print("missing quarters:", sorted(set(all_quarters) - set(quarter.unique())))
df["quarter"].value_counts(sort=False).to_frame("cases").T

## 4. Target: hospitalization or death

FDA codes the seriousness flags as `1` = yes, `2` = no. What a **blank** means depends on the quarter:

- **Up to 2021Q3** FDA only ever wrote `1`. A blank means **no**.
- **From 2021Q4** FDA writes both `1` and `2`. A blank means **unknown**.
- A **non-serious** report cannot involve death or hospitalization, so a blank there means **no**.

We find the "yes-only" quarters from the data instead of hard-coding the date.

`y_hosp_or_death` is `True` if either flag is `True`, `False` if both are `False`, and `<NA>` (unknown) otherwise.
Unknown rows are kept; notebook 3 drops them before modeling.

In [ ]:
def yes_no(column):
    """'1' -> True, '2' -> False, anything else -> <NA>."""
    return df[column].map({"1": True, "2": False}).astype("boolean")


death = yes_no("seriousnessdeath")
hosp = yes_no("seriousnesshospitalization")

# quarters where no flag is ever "2"
uses_no = (death.eq(False) | hosp.eq(False)).groupby(df["quarter"], observed=True).any()
yes_only_quarters = uses_no.index[~uses_no].tolist()
print("quarters that only record 'yes':", yes_only_quarters)

blank_means_no = df["serious"].eq("2") | df["quarter"].isin(yes_only_quarters)
death = death.mask(death.isna() & blank_means_no, False)
hosp = hosp.mask(hosp.isna() & blank_means_no, False)

df["is_serious"] = yes_no("serious")
df["death"] = death
df["hospitalization"] = hosp
df["y_hosp_or_death"] = death | hosp  # True | <NA> = True, False | <NA> = <NA>

print(df["y_hosp_or_death"].value_counts(dropna=False, normalize=True).round(3))
log_step("rebuild_target", len(df), len(df), f"{df['y_hosp_or_death'].isna().sum():,} unknown, kept")

## 5. Patient: sex, age group, weight

Age groups follow the ICH E2B bands: neonate < 28 days, infant < 2, child < 12, adolescent < 18, adult < 65,
elderly ≥ 65. Age 0 is kept (babies under one year are often coded as 0).

In [ ]:
df["sex"] = df["patientsex"].map({"1": "male", "2": "female"}).astype("category")  # '0' -> missing

AGE_GROUPS = ["neonate", "infant", "child", "adolescent", "adult", "elderly"]
from_age = pd.cut(df["age_years"], [0, 28 / 365.25, 2, 12, 18, 65, np.inf], labels=AGE_GROUPS, right=False)
from_code = df["patientagegroup"].map(dict(zip("123456", AGE_GROUPS))).astype(from_age.dtype)  # E2B codes 1-6
df["age_group"] = from_age.fillna(from_code)

weight = df["patientweight_kg"]
impossible = (weight < WEIGHT_MIN) | (weight > WEIGHT_MAX) | ((df["age_years"] < 2) & (weight > 40))
df["weight_kg"] = weight.mask(impossible)

print(f"age group known: {df['age_years'].notna().mean():.1%} from age -> {df['age_group'].notna().mean():.1%} with code")
print(f"impossible weights removed: {impossible.sum():,}")
log_step("patient_fields", len(df), len(df), f"{impossible.sum():,} weights -> missing")

## 6. Report fields and dates

- `receive_date`: when FDA **first** received the case.
- `receipt_date`: when FDA received **this version** of it.
- `is_legacy_case`: first received before 2020. These old cases appear only because a follow-up arrived later.

In [ ]:
REPORT_TYPE = {"1": "spontaneous", "2": "study", "3": "other", "4": "not_available"}
REPORTER = {"1": "physician", "2": "pharmacist", "3": "other_hcp", "4": "lawyer", "5": "consumer"}
EXPEDITED = {"1": "expedited_15day", "2": "non_expedited", "3": "direct", "4": "bsr", "5": "other"}

df["report_type"] = df["reporttype"].map(REPORT_TYPE).astype("category")
df["reporter"] = df["qualification"].map(REPORTER).astype("category")
df["expedited"] = df["fulfillexpeditecriteria"].map(EXPEDITED).astype("category")
df["occur_country"] = df["occurcountry"]

df["receive_date"] = df["receivedate_parsed"]
df["receipt_date"] = pd.to_datetime(df["receiptdate"].astype(str), format="%Y%m%d", errors="coerce")
df["is_legacy_case"] = df["receive_date"] < "2020-01-01"

df["reporter"].value_counts(dropna=False).to_frame("cases").T

## 7. Drop cases with no suspect drug

Without a suspect drug there is nothing to link the reaction to.

In [ ]:
before = len(df)
df = df[df["n_suspect_drugs"] > 0]
log_step("drop_no_suspect_drug", before, len(df))

## 8. Duplicates

Two cases count as identical when they match on: sex, age, weight, country, month received, and the exact drug and
reaction lists. Only cases with known sex, age and country are compared.

- **Group of 2–5 identical cases** (with ≥ 2 reactions each): duplicates. Keep the newest, drop the rest.
- **Group of more than 5**: a *mass report* (templated reports, e.g. from lawsuits). Keep, but flag `is_mass_report`.

Single-reaction cases are not treated as duplicates: profiles like "DUPILUMAB / DRUG INEFFECTIVE" are common
among different patients.

First we read the drug and reaction lists in batches and turn each case's pair of lists into one number (a hash),
so we can compare cases without holding the long text in memory.

In [ ]:
list_hash = []
for batch in pq.ParquetFile(INPUT).iter_batches(BATCH_SIZE, columns=["safetyreportid", "suspect_ingredients", "reaction_pts"]):
    b = batch.to_pandas()
    h = pd.util.hash_pandas_object(b[["suspect_ingredients", "reaction_pts"]], index=False)
    list_hash.append(pd.Series(h.values, index=b["safetyreportid"].astype("int64")))
list_hash = pd.concat(list_hash)

df = df.sort_values("receipt_date")  # newest copy last in each group

df["receive_month"] = df["receive_date"].dt.year * 100 + df["receive_date"].dt.month
df["lists_hash"] = df["safetyreportid"].map(list_hash)
match_cols = ["sex", "age_years", "weight_kg", "occur_country", "receive_month", "lists_hash"]
can_match = df[["sex", "age_years", "occur_country"]].notna().all(axis=1)

df["dup_key"] = pd.Series(pd.util.hash_pandas_object(df[match_cols], index=False).values, index=df.index,
                          dtype="UInt64").where(can_match)
group_size = df.groupby("dup_key")["dup_key"].transform("size").fillna(1)

df["is_mass_report"] = group_size > DUP_MAX_GROUP
small_group = group_size.between(2, DUP_MAX_GROUP) & (df["n_reactions"] >= 2)

is_duplicate = small_group & df.duplicated("dup_key", keep="last")

before = len(df)
df = df[~is_duplicate]
log_step("drop_duplicates", before, len(df), "groups of 2-5 identical cases, newest kept")
log_step("flag_mass_reports", len(df), len(df), f"{df['is_mass_report'].sum():,} flagged, kept")

## 9. Drug and reaction tables

Pass over the list. Each `" | "`-separated list becomes one row per item.

Ingredient names are standardized:
- combination products are split: `DOLUTEGRAVIR SODIUM\LAMIVUDINE` → two ingredients
- salt words at the end are removed: `RANITIDINE HYDROCHLORIDE` → `RANITIDINE`
- unless only a mineral would be left: `MAGNESIUM SULFATE` stays `MAGNESIUM SULFATE`

In [ ]:
SALT_WORDS = {
    "HYDROCHLORIDE", "HCL", "HYDROBROMIDE", "MESYLATE", "MESILATE", "DIMESYLATE", "MALEATE", "BESYLATE",
    "BESILATE", "TARTRATE", "BITARTRATE", "SUCCINATE", "FUMARATE", "CITRATE", "ACETATE", "SULFATE",
    "SULPHATE", "PHOSPHATE", "DIHYDRATE", "MONOHYDRATE", "TRIHYDRATE", "HEMIHYDRATE", "ANHYDROUS",
    "HYCLATE", "MALATE", "S-MALATE", "TOSYLATE", "DISODIUM", "SODIUM", "POTASSIUM", "CALCIUM", "MAGNESIUM",
    "DIPROPIONATE", "PROPIONATE", "FUROATE", "XINAFOATE", "BROMIDE", "LACTATE", "GLUCONATE", "NITRATE",
}
MINERALS = {"SODIUM", "POTASSIUM", "CALCIUM", "MAGNESIUM", "IRON", "ZINC", "FERROUS", "LITHIUM"}


def standardize(name):
    words = name.replace(",", " ").split()
    while len(words) > 1 and words[-1] in SALT_WORDS:
        words.pop()
    short = " ".join(words)
    return name if short in MINERALS else short


def explode(frame, column, new_name, sep):
    """One row per item of a separator-joined text column."""
    out = frame[["safetyreportid", column]].dropna()
    out[new_name] = out[column].str.split(sep, regex=False)
    out = out.explode(new_name)
    out[new_name] = out[new_name].str.strip()
    return out.loc[out[new_name] != "", ["safetyreportid", new_name]]


ING_DIR, RXN_DIR = OUT / "case_ingredients", OUT / "case_reactions"
for folder in (ING_DIR, RXN_DIR):
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir()

kept_ids = df["safetyreportid"].to_numpy()
ingredient_counts = []
batches = pq.ParquetFile(INPUT).iter_batches(BATCH_SIZE, columns=["safetyreportid", "suspect_ingredients", "reaction_pts"])
for i, batch in enumerate(batches):
    b = batch.to_pandas()
    b["safetyreportid"] = b["safetyreportid"].astype("int64")
    b = b[b["safetyreportid"].isin(kept_ids)]

    products = explode(b, "suspect_ingredients", "product", " | ")
    ing = explode(products, "product", "ingredient", "\\")
    names = {name: standardize(name) for name in ing["ingredient"].unique()}
    ing["ingredient"] = ing["ingredient"].map(names)
    ing = ing.drop_duplicates()

    rxn = explode(b, "reaction_pts", "reaction_pt", " | ").drop_duplicates()

    ing.to_parquet(ING_DIR / f"part_{i:03d}.parquet", index=False)
    rxn.to_parquet(RXN_DIR / f"part_{i:03d}.parquet", index=False)
    ingredient_counts.append(ing["safetyreportid"].value_counts())
    print(f"batch {i}: {len(b):,} cases -> {len(ing):,} ingredient rows, {len(rxn):,} reaction rows")

df["n_suspect_ingredients"] = df["safetyreportid"].map(pd.concat(ingredient_counts)).fillna(0).astype("int32")

## 10. Save

Keep the cleaned columns (raw code columns are dropped now that each has a readable version), run final checks,
and save.

In [ ]:
FINAL_COLS = [
    "safetyreportid", "safetyreportversion", "quarter",
    "y_hosp_or_death", "death", "hospitalization", "is_serious",
    "report_type", "reporter", "expedited", "occur_country",
    "receive_date", "receipt_date", "is_legacy_case",
    "sex", "age_years", "age_group", "weight_kg",
    "n_drugs", "n_suspect_drugs", "n_suspect_ingredients", "n_reactions",
    "is_mass_report",
]
clean = df[FINAL_COLS].sort_values("safetyreportid").reset_index(drop=True)

assert clean["safetyreportid"].is_unique
assert clean["weight_kg"].dropna().between(WEIGHT_MIN, WEIGHT_MAX).all()
assert not (clean["death"].eq(True) & clean["y_hosp_or_death"].eq(False)).any()

clean.to_parquet(OUT / "reports_clean.parquet", index=False)
log_step("save", len(clean), len(clean))
pd.DataFrame(log).to_csv(OUT / "cleaning_log.csv", index=False)

print(f"\nreports_clean: {clean.shape[0]:,} rows x {clean.shape[1]} columns")
pd.DataFrame(log)

## 11. Result

In [ ]:
print(clean.dtypes.to_string())
clean.head()

In [ ]:
clean.isna().mean().loc[lambda s: s > 0].sort_values(ascending=False).round(3).to_frame("share missing")

## Handoff for notebook 3

```
reports     = pd.read_parquet("data/clean/reports_clean.parquet")
ingredients = pd.read_parquet("data/clean/case_ingredients")   # safetyreportid, ingredient
reactions   = pd.read_parquet("data/clean/case_reactions")     # safetyreportid, reaction_pt
```

- Reaction terms such as `DEATH` and `HOSPITALISATION` restate the target.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=73434ccc-8a09-4578-aac6-0389f491d707' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>